In [1]:
# Import packages
import os
from pathlib import Path

os.chdir(Path.cwd().parent)
project_dir = Path("/home/mcaskey/10XvParse/")

import anndata as ad
import scanpy as sc
import pandas as pd
from scipy.stats import pearsonr

In [2]:
hto_data = ad.read_h5ad(project_dir / "Data/Analysis_2/10x_hashtags/kb_python/all_out/counts_unfiltered/adata.h5ad")
adata = ad.read_h5ad(project_dir / "Data/Analysis_2/10x/kb_python/all_out/counts_unfiltered/adata.h5ad")

In [ ]:
adata.obs

AnnData object with n_obs × n_vars = 1362559 × 78334

In [4]:
common = adata.obs_names.intersection(hto_data.obs_names)
adata = adata[common].copy()
hto_data = hto_data[common].copy()

# Put hashtag counts into adata.obs
hto_df = pd.DataFrame(
    hto_data.X.toarray() if hasattr(hto_data.X, "toarray") else hto_data.X,
    index=hto_data.obs_names,
    columns=hto_data.var_names
)

adata.obs = adata.obs.join(hto_df)

# Run HashSolo using the hashtag column names
sc.external.pp.hashsolo(adata, cell_hashing_columns=list(hto_data.var_names))

Please cite HashSolo paper:
https://www.cell.com/cell-systems/fulltext/S2405-4712(20)30195-2


In [5]:
demux_pd = adata.obs[["Classification"]]
print(demux_pd["Classification"].value_counts())

Classification
10x_A2     4601
10x_B2     3939
10x_B1     2137
Doublet    1856
10x_A1     1755
Name: count, dtype: int64


In [6]:
hto_config_dir = project_dir / "Configs/Analysis_2/10x"
Path(project_dir / hto_config_dir).mkdir( exist_ok=True)
demux_pd.to_csv(hto_config_dir / "hto_demultiplexed.csv")